# Modern (Simple English Wikipedia) – 02 Train & Generation

Fine-tune **SmolLM2-135M** on the Simple English Wikipedia articles cleaned in `01_data.ipynb`, save and reload the checkpoint, and compare generations before and after training.

**Run `01_data.ipynb` first:** it writes `modern_cleaning_output/cleaned_articles.jsonl`, which this notebook reads.

**Deliverables:** training-loss plot, saved checkpoint (reloaded and checked), two before/after examples.

## 1. Setup

In [ ]:
import math
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
DATA_PATH  = "modern_cleaning_output/cleaned_articles.jsonl"   # written by 01_data.ipynb
OUT_DIR    = Path("../../results/modern")
BLOCK      = 256       # tokens per training block
SEED       = 42

EPOCHS, BS, LR = 3, 16, 5e-5

PREFIX_1 = "The Amazon River is"
PREFIX_2 = "In computer science, a"

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the cleaned articles

In [ ]:
ds = load_dataset("json", data_files=DATA_PATH)["train"]
ds = ds.filter(lambda ex: len(ex["text"].strip()) > 0)
print("articles:", len(ds))

# The data team marked these flags as "needs review"; we leave those articles out.
REVIEW_FLAGS = {"empty_parentheses", "leftover_wiki_markup"}
before_filter = len(ds)
ds = ds.filter(lambda ex: not REVIEW_FLAGS.intersection(ex["flags"]))
print(f"dropped {before_filter - len(ds)} flagged articles -> {len(ds)} left")

print("\n" + ds[0]["title"] + "\n" + ds[0]["text"][:500])

In [ ]:
# Contiguous 90/10 split by article (no shuffle), so test articles are never seen in training.
data = ds.train_test_split(test_size=0.1, shuffle=False)
data

**Formatting notes**
- Text was cleaned by the data team (`modern_english_cleaning.py`): wiki markup, citations and back-matter sections removed.
- Articles already use `\n\n` between their own paragraphs, so we mark the end of each article with the `<|endoftext|>` (EOS) token instead.

## 3. Tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(device)

print("params:", model.num_parameters())
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id, "| pad:", tokenizer.pad_token, "| vocab:", tokenizer.vocab_size)

In [ ]:
# Quick look at how text becomes tokens.
example = data["train"][0]["text"][:200]
ids = tokenizer(example)["input_ids"]
print(list(zip(ids, tokenizer.convert_ids_to_tokens(ids)))[:20])
print("round trip ok:", tokenizer.decode(ids) == example)
print(f"chars per token: {len(example) / len(ids):.2f}")

## 4. Before training: generation

In [ ]:
gen_kwargs = dict(
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

def generate_text(m, prefix, seed=SEED):
    m.eval()
    with torch.inference_mode():
        enc = tokenizer(prefix, return_tensors="pt").to(device)
        torch.manual_seed(seed)
        out = m.generate(**enc, **gen_kwargs)
    return tokenizer.decode(out[0])

before_1 = generate_text(model, PREFIX_1)
before_2 = generate_text(model, PREFIX_2)
print(before_1, "\n" + "=" * 60 + "\n" + before_2)

## 5. Build 256-token blocks

In [ ]:
def stream_ids(texts):
    # tokenize each article and add EOS after it, so the model sees where articles end
    ids = []
    for t in texts:
        ids += tokenizer(t)["input_ids"] + [tokenizer.eos_token_id]
    return ids

def make_blocks(ids, block=BLOCK):
    n = len(ids) // block                      # leftover tail is dropped
    return [ids[i * block:(i + 1) * block] for i in range(n)]

def to_ds(blocks):
    # labels = copy of input_ids; the model shifts them internally
    return Dataset.from_dict({"input_ids": blocks, "labels": [b.copy() for b in blocks]})

train_ids = stream_ids(data["train"]["text"])
test_ids  = stream_ids(data["test"]["text"])

train_blocks, test_blocks = make_blocks(train_ids), make_blocks(test_ids)
assert all(len(b) == BLOCK for b in train_blocks + test_blocks)

lm_data = DatasetDict({
    "train": to_ds(train_blocks).shuffle(seed=SEED),   # shuffle train only
    "test":  to_ds(test_blocks),
})
print(f"train: {len(train_ids):,} tokens -> {len(train_blocks)} blocks")
print(f"test:  {len(test_ids):,} tokens -> {len(test_blocks)} blocks")

In [ ]:
print(tokenizer.decode(lm_data["train"][0]["input_ids"])[:600])

## 6. Fine-tune

In [ ]:
torch.manual_seed(SEED)

lm_data.set_format("torch", columns=["input_ids", "labels"])
train_loader = DataLoader(lm_data["train"], batch_size=BS, shuffle=True)
test_loader  = DataLoader(lm_data["test"],  batch_size=BS)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
sched = get_cosine_schedule_with_warmup(opt, int(0.03 * total_steps), total_steps)

@torch.no_grad()
def evaluate():
    model.eval()
    tot, n = 0.0, 0
    for b in test_loader:
        ids, lab = b["input_ids"].to(device), b["labels"].to(device)
        tot += model(input_ids=ids, labels=lab).loss.item() * ids.size(0)
        n += ids.size(0)
    model.train()
    return tot / n

In [ ]:
v = evaluate()
print(f"before training | val loss {v:.4f} | ppl {math.exp(v):.2f}")

log_steps, log_losses, val_losses = [], [], []
step = 0
model.train()
for epoch in range(1, EPOCHS + 1):
    for b in train_loader:
        ids, lab = b["input_ids"].to(device), b["labels"].to(device)
        loss = model(input_ids=ids, labels=lab).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)

        step += 1
        if step == 1 or step % 10 == 0:
            log_steps.append(step); log_losses.append(loss.item())
            print(f"step {step}/{total_steps} | loss {loss.item():.4f}")

    v = evaluate()
    val_losses.append(v)
    print(f"epoch {epoch} | val loss {v:.4f} | ppl {math.exp(v):.2f}")
    ckpt_dir = OUT_DIR / f"ckpt_epoch{epoch}"
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)

In [ ]:
plt.plot(log_steps, log_losses)
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("SmolLM2-135M on Simple English Wikipedia")
plt.savefig(OUT_DIR / "train_loss.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save and reload check

Load the last checkpoint from disk as a brand-new model and generate with the same prefix and seed as the model still in memory. If saving worked, the two outputs are identical.

In [ ]:
reloaded = AutoModelForCausalLM.from_pretrained(OUT_DIR / f"ckpt_epoch{EPOCHS}").to(device)

from_memory = generate_text(model, PREFIX_1)
from_disk   = generate_text(reloaded, PREFIX_1)
print("reload matches in-memory model:", from_memory == from_disk)
del reloaded

## 8. Load the best checkpoint

In [ ]:
best_epoch = min(range(len(val_losses)), key=lambda i: val_losses[i]) + 1
ckpt = OUT_DIR / f"ckpt_epoch{best_epoch}"
print("best:", ckpt, "| val loss", round(val_losses[best_epoch - 1], 4))

tuned = AutoModelForCausalLM.from_pretrained(ckpt).to(device)

## 9. After training: generation

In [ ]:
after_1 = generate_text(tuned, PREFIX_1)
after_2 = generate_text(tuned, PREFIX_2)

for name, b, a in [("prefix 1", before_1, after_1), ("prefix 2", before_2, after_2)]:
    print(f"##### {name} — BEFORE #####\n{b}\n")
    print(f"##### {name} — AFTER #####\n{a}\n")

## 10. Observations

- **Data:** 500 Simple English Wikipedia articles cleaned by the data team; ... flagged articles dropped; articles separated by EOS. After training the model may end an article with `<|endoftext|>`, so some outputs stop before 120 tokens.
- **Settings:** seed 42, `max_new_tokens=120`, temperature 0.8, top-p 0.9; identical before and after.
- **Loss:** before training val loss = ..., best epoch = ..., val loss = ...
- **Reload check:** reloaded checkpoint matches the in-memory model: ...
- **Before vs after:** ...
- **Partners:** my partner left the team, so I completed this task alone and can explain every step.
- **Limitations:** one sample per prefix (single seed).